In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
ENSEMBLE_PATH  = "ensemble_predictions.csv"
SENTIMENT_PATH = "stock_sentiment_history.csv"

RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]

MANUAL_WEIGHTS = [        # (w_ensemble, w_sentiment)
    (0.4, 0.6),
    (0.5, 0.5),
    (0.7, 0.3),
    (0.8, 0.2),
]

# ─────────────────────────────────────────────
# 1.  METRICS
# ─────────────────────────────────────────────

def sharpe(returns: np.ndarray, annualize: int = 252) -> float:
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(annualize)

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    signal_returns = np.sign(y_pred) * y_true
    return {
        "DirAcc":  directional_accuracy(y_true, y_pred),
        "Sharpe":  sharpe(signal_returns),
        "RMSE":    rmse(y_true, y_pred),
        "MeanRet": signal_returns.mean(),
        "RetStd":  signal_returns.std(),
    }


# ─────────────────────────────────────────────
# 2.  LOAD & ALIGN DATA
# ─────────────────────────────────────────────

def load_and_align(ensemble_path: str, sentiment_path: str) -> pd.DataFrame:
    ens = pd.read_csv(ensemble_path, parse_dates=["date"])
    ens = ens[["date", "fold_id", "y_true", "ensemble_A"]].rename(
        columns={"ensemble_A": "ensemble_pred"}
    )

    sent = pd.read_csv(sentiment_path, parse_dates=["Date"])
    sent = sent.rename(columns={"Date": "date"})
    sent = sent[["date", "Final_Sentiment", "Article_Count"]]

    merged = ens.merge(sent, on="date", how="inner")
    merged = merged.sort_values(["fold_id", "date"]).reset_index(drop=True)

    n_dropped = len(ens) - len(merged)
    print(f"  Ensemble rows:   {len(ens):,}")
    print(f"  Sentiment rows:  {len(sent):,}")
    print(f"  Merged rows:     {len(merged):,}  ({n_dropped} ensemble dates dropped — no sentiment match)")

    return merged


# ─────────────────────────────────────────────
# 3.  NORMALIZE FEATURES
# ─────────────────────────────────────────────

def normalize_features(df: pd.DataFrame) -> tuple[pd.DataFrame, StandardScaler]:
    """
    Z-score normalize both features independently so Ridge operates on a
    level playing field. Raw (unscaled) signals are kept for manual fusion.
    """
    df = df.copy()
    scaler = StandardScaler()

    features = df[["ensemble_pred", "Final_Sentiment"]].values
    df[["ensemble_pred_scaled", "sentiment_scaled"]] = scaler.fit_transform(features)

    print(f"\n── Feature normalization ──")
    print(f"  ensemble_pred   — mean: {df['ensemble_pred'].mean():.6f},  std: {df['ensemble_pred'].std():.6f}")
    print(f"  Final_Sentiment — mean: {df['Final_Sentiment'].mean():.6f},  std: {df['Final_Sentiment'].std():.6f}")

    return df, scaler


# ─────────────────────────────────────────────
# 4.  WALK-FORWARD RIDGE FUSION
# ─────────────────────────────────────────────

def walk_forward_ridge(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each fold_id, train Ridge on all *previous* folds and predict on
    the current fold — preserving temporal ordering and preventing leakage.

    Features : [ensemble_pred_scaled, sentiment_scaled]
    Target   : y_true
    """
    df = df.copy()
    df["ridge_pred"]         = np.nan
    df["ridge_alpha_chosen"] = np.nan

    folds        = sorted(df["fold_id"].unique())
    feature_cols = ["ensemble_pred_scaled", "sentiment_scaled"]

    if len(folds) < 2:
        raise ValueError("Need at least 2 folds for walk-forward validation.")

    print(f"\n── Walk-forward Ridge fusion ──")
    print(f"  Folds found: {folds}")

    for i, fold in enumerate(folds):
        if i == 0:
            mask_test = df["fold_id"] == fold
            df.loc[mask_test, "ridge_pred"] = df.loc[mask_test, "ensemble_pred"]
            print(f"  Fold {fold}: no training history → fallback to ensemble_pred")
            continue

        train_folds = folds[:i]
        mask_train  = df["fold_id"].isin(train_folds)
        mask_test   = df["fold_id"] == fold

        X_train = df.loc[mask_train, feature_cols].values
        y_train = df.loc[mask_train, "y_true"].values
        X_test  = df.loc[mask_test,  feature_cols].values

        model = RidgeCV(alphas=RIDGE_ALPHAS, fit_intercept=True)
        model.fit(X_train, y_train)

        df.loc[mask_test, "ridge_pred"]         = model.predict(X_test)
        df.loc[mask_test, "ridge_alpha_chosen"] = model.alpha_

        print(f"  Fold {fold}: trained on folds {train_folds}  |  "
              f"α={model.alpha_}  |  "
              f"coefs=[ens={model.coef_[0]:+.4f}, sent={model.coef_[1]:+.4f}]  |  "
              f"intercept={model.intercept_:+.4f}  |  "
              f"n_test={mask_test.sum()}")

    return df


# ─────────────────────────────────────────────
# 5.  MANUAL WEIGHT FUSION
# ─────────────────────────────────────────────

def apply_manual_weights(df: pd.DataFrame) -> pd.DataFrame:
    """
    Simple weighted average on the *scaled* features so manual blends are
    directly comparable to Ridge (which also operates on scaled inputs).
    """
    df = df.copy()

    for w_e, w_s in MANUAL_WEIGHTS:
        label = f"manual_{int(w_e*100)}_{int(w_s*100)}"
        df[label] = w_e * df["ensemble_pred_scaled"] + w_s * df["sentiment_scaled"]
        print(f"  {label}: w_ensemble={w_e}, w_sentiment={w_s}")

    return df

# ─────────────────────────────────────────────
# 6.  MAIN
# ─────────────────────────────────────────────

def main():
    print("Loading and aligning data …")
    df = load_and_align(ENSEMBLE_PATH, SENTIMENT_PATH)

    df, scaler = normalize_features(df)
    df = walk_forward_ridge(df)

    print(f"\n── Manual weight fusion ──")
    df = apply_manual_weights(df)

    y_true = df["y_true"].values

    # ── Collect all variants ──
    results = {"ensemble_only": compute_metrics(y_true, df["ensemble_pred"].values)}

    results["ridge_fusion"] = compute_metrics(y_true, df["ridge_pred"].values)

    for w_e, w_s in MANUAL_WEIGHTS:
        label = f"manual_{int(w_e*100)}_{int(w_s*100)}"
        results[label] = compute_metrics(y_true, df[label].values)

    metrics_df = (
        pd.DataFrame(results).T
        .rename_axis("Variant")
        .sort_values("DirAcc", ascending=False)
    )

    # ── Tag variant type for readability ──
    def tag(name):
        if name == "ensemble_only": return "baseline"
        if name == "ridge_fusion":  return "ridge"
        return "manual"

    metrics_df.insert(0, "Type", [tag(n) for n in metrics_df.index])

    print(f"\n{'='*65}")
    print(f"  FUSION RESULTS  (ranked by DirAcc)")
    print(f"{'='*65}")
    print()
    print(metrics_df.round(5).to_string())

    winner = metrics_df.index[0]
    wm     = metrics_df.loc[winner]
    print(f"\n  ► WINNER: [{winner}]  ({wm['Type']})")
    print(f"    DirAcc={wm['DirAcc']:.4f}  Sharpe={wm['Sharpe']:+.4f}  RMSE={wm['RMSE']:.5f}")
    print(f"{'='*65}")

    # ── Save ──
    manual_cols = [f"manual_{int(w_e*100)}_{int(w_s*100)}" for w_e, w_s in MANUAL_WEIGHTS]
    out_cols = [
        "date", "fold_id", "y_true",
        "ensemble_pred", "Final_Sentiment", "Article_Count",
        "ensemble_pred_scaled", "sentiment_scaled",
        "ridge_pred", "ridge_alpha_chosen",
        *manual_cols,
    ]
    df[out_cols].to_csv("late_fusion_predictions.csv", index=False)
    metrics_df.to_csv("late_fusion_metrics.csv")

    print("\n  Saved: late_fusion_predictions.csv")
    print("  Saved: late_fusion_metrics.csv")

    return df, metrics_df

if __name__ == "__main__":
    df, metrics_df = main()

Loading and aligning data …
  Ensemble rows:   727
  Sentiment rows:  1,522
  Merged rows:     768  (-41 ensemble dates dropped — no sentiment match)

── Feature normalization ──
  ensemble_pred   — mean: 0.001374,  std: 0.002536
  Final_Sentiment — mean: 0.046955,  std: 0.376008

── Walk-forward Ridge fusion ──
  Folds found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]
  Fold 1: no training history → fallback to ensemble_pred
  Fold 2: trained on folds [1]  |  α=100.0  |  coefs=[ens=+0.0001, sent=+0.0017]  |  intercept=-0.0050  |  n_test=23
  Fold 3: trained on folds [1, 2]  |  α=100.0  |  coefs=[ens=+0.0006, sent=+0.0016]  |  intercept=+0.0019  |  n_test=19
  Fold 4: trained on folds [1, 2, 3]  |  α=100.0  |  coefs=[ens=+0.0006, sent=-0.0003]  |  intercept=+0.0018  |  n_test=22
  Fold 5: trained on folds [1, 2, 3, 4]  |  α=100.0  |  coefs=[ens=+0.0010, sent=+0.0000]  |  intercept=+0.0034  |  n_tes